In [1]:
import os
import json
import mlflow
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from workspace.sources.experiments.metrics import Metric
from workspace.sources.experiments.visualizations.utils import METRICS_PLOT_NAMES_MAPPING
from workspace.sources.local_datasets.dataset import Dataset
from sklearn.metrics import confusion_matrix
from workspace.sources.experiments.metrics import Precision, Loss, standard_evaluation_metrics, standard_metrics

from workspace.sources.experiments.visualizations.plots import plot_confusion_matrix, plot_roc_curve

mlflow.set_tracking_uri('../mlruns')

In [5]:
experiments = ['prefinal-bert-base-uncased', 'prefinal-distillibert-base-uncased']
client = mlflow.tracking.MlflowClient()

### ReCOVery Dataset

In [6]:
dataset_name = 'recovery'

#### Best Models Table

In [7]:
experiments_runs = dict()
for experiment in experiments:
    experiment_runs = client.search_runs(
        experiment_ids=[client.get_experiment_by_name(experiment).experiment_id],
        filter_string=f"params.dataset_name = '{dataset_name}'"
    )
    experiments_runs[experiment] = experiment_runs

In [8]:
def pick_best_run_by_metric(runs, metric, second_metric=Loss):
    """Pick the best run based on given metric and its loss value."""
    metrics_keys = [f'test_{metric.name}_by_{metric.name}', f'test_{second_metric.name}_by_{metric.name}']
    metrics_greater_is_better = [metric.greater_is_better,
                                 second_metric.greater_is_better]
    metrics = zip(metrics_keys, metrics_greater_is_better)
    # Filter runs that have required metrics
    valid_runs = [run for run in runs
                  if all(key in run.data.metrics for key in metrics_keys)]

    if not valid_runs:
        return None

    # Define sort key function
    def _get_run_sort_key(run):
        return tuple(run.data.metrics[m_key] if m_gib else -run.data.metrics[m_key] for m_key, m_gib in metrics)

    # Return run with best metrics
    return max(valid_runs, key=_get_run_sort_key)


In [9]:

best_models_by_metrics = dict()
for metric in standard_evaluation_metrics:
    best_models_by_metrics[metric.name] = dict()
    for experiment, runs in experiments_runs.items():
        best_run = pick_best_run_by_metric(runs, metric)
        if best_run is not None:
            best_models_by_metrics[metric.name][experiment] = best_run


In [10]:
def create_metrics_comparison_df(metric, best_modesl_by_metrics_data):
    metrics_df_data = []
    experiments_data = best_modesl_by_metrics_data[metric.name]
    for experiment_name, run in experiments_data.items():
        experiment_data = {
            'Experiment': experiment_name,
            'Run Signature': run.data.params['run_signature'],
            'Evaluation Metric': metric.name,
            'Model': run.data.params['model_name']
        }
        metrics_data = {METRICS_PLOT_NAMES_MAPPING[m.name]: run.data.metrics[f'test_{m.name}_by_{metric.name}']
                        for m in standard_metrics
                        if f'test_{m.name}_by_{metric.name}' in run.data.metrics}
        params_data = {
            'best_epoch': run.data.metrics[f'best_epoch_by_{metric.name}'],
        }
        data = {**experiment_data, **metrics_data, **params_data}
        metrics_df_data.append(data)

    return pd.DataFrame(metrics_df_data)


best_models_by_loss = create_metrics_comparison_df(Loss, best_models_by_metrics)
best_models_by_loss

,Experiment,Run Signature,Evaluation Metric,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,FPR,FNR,best_epoch
0,prefinal-bert-base-uncased,"model(mn=bert-base-uncased,mmn=loss,e=10,bs=8,...",loss,bert-base-uncased,0.633333,0.633333,1.0,0.77551,0.722488,1.0,0.0,2.0
1,prefinal-distillibert-base-uncased,"model(mn=distilbert-base-uncased,mmn=loss,e=10...",loss,distilbert-base-uncased,0.666667,0.666667,1.0,0.80000,0.755000,1.0,0.0,2.0


In [15]:
def export_best_models_to_latex(metrics_df: pd.DataFrame, output_path: str = 'assets/tables/') -> None:
    by_metric = metrics_df['Evaluation Metric'].iloc[0]
    experiment_columns = ['Model']
    metrics_columns = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'FPR', 'FNR']
    selected_columns = experiment_columns + metrics_columns
    selection_df = metrics_df[selected_columns]
    latex_table = selection_df.to_latex(index=False, escape=False, float_format='%.3f')

    output_filepath = os.path.join(output_path, f'best_models_table_by_{by_metric}.txt')

    os.makedirs(os.path.dirname(output_filepath), exist_ok=True)
    with open(output_filepath, 'w') as f:
        f.write(latex_table)


export_best_models_to_latex(best_models_by_loss)
